# Market-Driven Skill Recommendation System for Developers
## Colab Pipeline: Data Validation → Cleaning → Standardization → Train/Test Preparation

Notebook này triển khai toàn bộ **data pipeline A-Z** cho đề tài:

**Market-Driven Skill Recommendation System for Developers using Hybrid Association Rule Mining and Graph Embedding**

Mục tiêu:
1. Đọc và kiểm tra dataset Stack Overflow Survey 2025  
2. Lọc population phù hợp với research scope  
3. Chuẩn hóa các trường kỹ năng multi-select  
4. Tạo input cho FP-Growth, Node2Vec và evaluation  
5. Tách train/test đúng logic để tránh data leakage  
6. Xuất các file sạch sẵn sàng cho bước training model

## 0. Cài thư viện
Chạy cell này trước trên Google Colab.

In [1]:
!pip -q install pandas numpy scikit-learn mlxtend node2vec networkx==3.3



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Import thư viện

In [2]:
import os
import re
import json
import math
import itertools
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx

from sklearn.model_selection import train_test_split
from mlxtend.preprocessing import TransactionEncoder

## 3. Đọc dữ liệu
- `low_memory=False` để tránh warning type inference
- dùng `dtype=str` để pipeline ổn định hơn trong giai đoạn clean

In [4]:
PUBLIC_CSV = "survey_results_public.csv"
SCHEMA_CSV = "survey_results_schema.csv"

assert os.path.exists(PUBLIC_CSV), f"Không tìm thấy {PUBLIC_CSV}"
assert os.path.exists(SCHEMA_CSV), f"Không tìm thấy {SCHEMA_CSV}"

df = pd.read_csv(PUBLIC_CSV, low_memory=False, dtype=str)
schema_df = pd.read_csv(SCHEMA_CSV, low_memory=False, dtype=str)

print("Public data shape:", df.shape)
print("Schema shape:", schema_df.shape)
display(df.head(3))

Public data shape: (49191, 172)
Schema shape: (139, 6)


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256,10
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413,9
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061,8


## 4. Validate schema bắt buộc

Đây là bước quan trọng để tránh pipeline bị vỡ ở các cell sau.

In [5]:
REQUIRED_COLUMNS = [
    "ResponseId",
    "MainBranch",
    "Employment",
    "WorkExp",
    "DevType",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",
    "PlatformHaveWorkedWith",
    "PlatformWantToWorkWith",
    "WebframeHaveWorkedWith",
    "WebframeWantToWorkWith",
    "DevEnvsHaveWorkedWith",
    "DevEnvsWantToWorkWith",
    "AIModelsHaveWorkedWith",
    "AIModelsWantToWorkWith",
]

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Thiếu cột bắt buộc: {missing_cols}")

print("Tất cả cột bắt buộc đều tồn tại.")

Tất cả cột bắt buộc đều tồn tại.


## 5. Khám phá nhanh dữ liệu thô

In [6]:
summary = pd.DataFrame({
    "column": REQUIRED_COLUMNS,
    "missing_count": [df[c].isna().sum() for c in REQUIRED_COLUMNS],
    "missing_ratio": [df[c].isna().mean() for c in REQUIRED_COLUMNS]
}).sort_values("missing_ratio", ascending=False)

display(summary)
print("Tổng số dòng:", len(df))
print("Số lượng cột:", len(df.columns))

,column,missing_count,missing_ratio
16,AIModelsWantToWorkWith,37346,0.759204
15,AIModelsHaveWorkedWith,32910,0.669025
12,WebframeWantToWorkWith,31552,0.641418
10,PlatformWantToWorkWith,29776,0.605314
8,DatabaseWantToWorkWith,29458,0.598849
14,DevEnvsWantToWorkWith,28577,0.580940
11,WebframeHaveWorkedWith,26199,0.532597
9,PlatformHaveWorkedWith,24933,0.506861
7,DatabaseHaveWorkedWith,23641,0.480596
13,DevEnvsHaveWorkedWith,23172,0.471062


Tổng số dòng: 49191
Số lượng cột: 172


## 6. Xác định population nghiên cứu

### Base population để train:
- `MainBranch == "I am a developer by profession"`
- `Employment` chứa một trong các trạng thái nghề nghiệp thực tế:
  - Employed
  - Independent contractor / freelancer / self-employed

### Junior subset để evaluate riêng:
- `WorkExp` từ 0 đến 3 năm

In [7]:
def parse_workexp(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    # đa số WorkExp là số nguyên dạng chuỗi
    m = re.search(r"\d+", x)
    return float(m.group()) if m else np.nan

df["WorkExp_num"] = df["WorkExp"].apply(parse_workexp)

df["is_developer_profession"] = df["MainBranch"].fillna("").eq("I am a developer by profession")

employment_text = df["Employment"].fillna("")
df["is_employed_or_self_employed"] = employment_text.str.contains(
    "Employed|Independent contractor, freelancer, or self-employed",
    case=False,
    regex=True
)

df["is_base_population"] = df["is_developer_profession"] & df["is_employed_or_self_employed"]
df["is_junior_0_3y"] = df["WorkExp_num"].between(0, 3, inclusive="both")

print("Toàn bộ dữ liệu:", len(df))
print("Developer by profession:", int(df["is_developer_profession"].sum()))
print("Employed/self-employed:", int(df["is_employed_or_self_employed"].sum()))
print("Base population:", int(df["is_base_population"].sum()))
print("Junior 0-3 years (toàn bộ):", int(df["is_junior_0_3y"].sum()))
print("Junior 0-3 years trong base population:", int((df["is_base_population"] & df["is_junior_0_3y"]).sum()))

Toàn bộ dữ liệu: 49191
Developer by profession: 37467
Employed/self-employed: 42685
Base population: 35321
Junior 0-3 years (toàn bộ): 7474
Junior 0-3 years trong base population: 5002


## 7. Chỉ giữ population phù hợp

In [8]:
base_df = df.loc[df["is_base_population"]].copy()
base_df.reset_index(drop=True, inplace=True)

print("Base population shape:", base_df.shape)
display(base_df[["ResponseId", "MainBranch", "Employment", "WorkExp", "WorkExp_num"]].head())

Base population shape: (35321, 177)


,ResponseId,MainBranch,Employment,WorkExp,WorkExp_num
0,1,I am a developer by profession,Employed,8,8.0
1,2,I am a developer by profession,Employed,2,2.0
2,3,I am a developer by profession,"Independent contractor, freelancer, or self-em...",10,10.0
3,4,I am a developer by profession,Employed,4,4.0
4,5,I am a developer by profession,"Independent contractor, freelancer, or self-em...",21,21.0


## 8. Chuẩn hóa skill columns

### Nguyên tắc
- split theo dấu `;`
- trim khoảng trắng
- loại token rỗng
- loại duplicate trong cùng một user
- thêm prefix namespace để tránh trùng tên giữa các nhóm skill

Ví dụ:
- `Python` → `Language:Python`
- `Docker` → `Platform:Docker`

In [9]:
SKILL_COLUMN_MAP = {
    "LanguageHaveWorkedWith": "Language",
    "LanguageWantToWorkWith": "Language",
    "DatabaseHaveWorkedWith": "Database",
    "DatabaseWantToWorkWith": "Database",
    "PlatformHaveWorkedWith": "Platform",
    "PlatformWantToWorkWith": "Platform",
    "WebframeHaveWorkedWith": "Webframe",
    "WebframeWantToWorkWith": "Webframe",
    "DevEnvsHaveWorkedWith": "DevEnv",
    "DevEnvsWantToWorkWith": "DevEnv",
    "AIModelsHaveWorkedWith": "AIModel",
    "AIModelsWantToWorkWith": "AIModel",
}

HAVE_COLS = [
    "LanguageHaveWorkedWith",
    "DatabaseHaveWorkedWith",
    "PlatformHaveWorkedWith",
    "WebframeHaveWorkedWith",
    "DevEnvsHaveWorkedWith",
    "AIModelsHaveWorkedWith",
]

WANT_COLS = [
    "LanguageWantToWorkWith",
    "DatabaseWantToWorkWith",
    "PlatformWantToWorkWith",
    "WebframeWantToWorkWith",
    "DevEnvsWantToWorkWith",
    "AIModelsWantToWorkWith",
]

INVALID_TOKENS = {"", "nan", "none", "null", "na", "n/a"}

def tokenize_multiselect(value):
    if pd.isna(value):
        return []
    parts = [p.strip() for p in str(value).split(";")]
    parts = [p for p in parts if p and p.strip().lower() not in INVALID_TOKENS]
    return parts

def prefix_tokens(tokens, prefix):
    return [f"{prefix}:{token}" for token in tokens]

def unique_preserve_order(items):
    seen = set()
    result = []
    for x in items:
        if x not in seen:
            seen.add(x)
            result.append(x)
    return result

def build_skill_list(row, columns):
    all_skills = []
    for col in columns:
        prefix = SKILL_COLUMN_MAP[col]
        tokens = tokenize_multiselect(row[col])
        tokens = prefix_tokens(tokens, prefix)
        all_skills.extend(tokens)
    return unique_preserve_order(all_skills)

base_df["have_skills_raw"] = base_df.apply(lambda row: build_skill_list(row, HAVE_COLS), axis=1)
base_df["want_skills_raw"] = base_df.apply(lambda row: build_skill_list(row, WANT_COLS), axis=1)
base_df["target_skills_raw"] = base_df.apply(
    lambda row: [s for s in row["want_skills_raw"] if s not in set(row["have_skills_raw"])],
    axis=1
)

base_df["n_have_raw"] = base_df["have_skills_raw"].apply(len)
base_df["n_want_raw"] = base_df["want_skills_raw"].apply(len)
base_df["n_target_raw"] = base_df["target_skills_raw"].apply(len)

display(base_df[["ResponseId", "have_skills_raw", "want_skills_raw", "target_skills_raw"]].head(3))

,ResponseId,have_skills_raw,want_skills_raw,target_skills_raw
0,1,"[Language:Bash/Shell (all shells), Language:Da...",[Language:Dart],[]
1,2,"[Language:Java, Database:Dynamodb, Database:Mo...","[Language:Java, Language:Python, Language:Swif...","[Language:Python, Language:Swift]"
2,3,"[Language:Dart, Language:HTML/CSS, Language:Ja...","[Language:Dart, Language:HTML/CSS, Language:Ja...","[Database:Firebase Realtime Database, Platform..."


## 9. Loại những user không có input skills

Đây là bước bắt buộc cho recommender.

In [10]:
model_df = base_df.loc[base_df["n_have_raw"] > 0].copy()
model_df.reset_index(drop=True, inplace=True)

print("Base population trước lọc empty input:", len(base_df))
print("Model population sau lọc empty input:", len(model_df))

Base population trước lọc empty input: 35321
Model population sau lọc empty input: 24229


## 10. Thống kê vocabulary trước filtering

In [11]:
have_counter = Counter()
want_counter = Counter()

for skills in model_df["have_skills_raw"]:
    have_counter.update(skills)

for skills in model_df["want_skills_raw"]:
    want_counter.update(skills)

vocab_df = pd.DataFrame({
    "skill": sorted(set(have_counter.keys()) | set(want_counter.keys()))
})
vocab_df["have_count"] = vocab_df["skill"].map(have_counter).fillna(0).astype(int)
vocab_df["want_count"] = vocab_df["skill"].map(want_counter).fillna(0).astype(int)
vocab_df["have_freq"] = vocab_df["have_count"] / len(model_df)
vocab_df["want_freq"] = vocab_df["want_count"] / len(model_df)

display(vocab_df.sort_values("have_count", ascending=False).head(20))
print("Kích thước vocabulary trước filtering:", len(vocab_df))

,skill,have_count,want_count,have_freq,want_freq
93,Language:JavaScript,16417,8098,0.677576,0.334228
69,DevEnv:Visual Studio Code,15411,9799,0.636056,0.404433
91,Language:HTML/CSS,14994,8051,0.618845,0.332288
109,Language:SQL,14697,8754,0.606587,0.361303
126,Platform:Docker,14348,9928,0.592183,0.409757
105,Language:Python,12911,8629,0.532874,0.356143
41,Database:PostgreSQL,11887,9622,0.490610,0.397127
112,Language:TypeScript,11687,8236,0.482356,0.339923
76,Language:Bash/Shell (all shells),11615,6177,0.479384,0.254942
156,Platform:npm,11517,5334,0.475339,0.220149


Kích thước vocabulary trước filtering: 186


## 11. Long-tail filtering

Theo proposal, loại skill xuất hiện dưới **0.5%** population.

Ở đây dùng:
\[
freq(skill) = users\_having\_skill / N
\]

và giữ skill nếu:
\[
freq(skill) \ge 0.005
\]

In [12]:
MIN_FREQ = 0.005
MIN_HAVE_FREQ = MIN_FREQ
MIN_WANT_FREQ = MIN_FREQ

MIN_HAVE_COUNT = math.ceil(MIN_HAVE_FREQ * len(model_df))
MIN_WANT_COUNT = math.ceil(MIN_WANT_FREQ * len(model_df))

# Gi? skill n?u skill ?? ?? ph? bi?n ? current skills HO?C wanted skills.
# ?i?u n?y tr?nh lo?i b? c?c skill m?i n?i: ?t ng??i ?? d?ng nh?ng nhi?u ng??i mu?n h?c.
skills_to_keep = set(
    vocab_df.loc[
        (vocab_df["have_count"] >= MIN_HAVE_COUNT)
        | (vocab_df["want_count"] >= MIN_WANT_COUNT),
        "skill",
    ]
)

print("Population size:", len(model_df))
print("Min have-count threshold:", MIN_HAVE_COUNT)
print("Min want-count threshold:", MIN_WANT_COUNT)
print("S? skill ???c gi? l?i:", len(skills_to_keep))
print("Skills gi? nh? wanted-skill signal:", int(((vocab_df["have_count"] < MIN_HAVE_COUNT) & (vocab_df["want_count"] >= MIN_WANT_COUNT)).sum()))


Population size: 24229
Min have-count threshold: 122
Min want-count threshold: 122
S? skill ???c gi? l?i: 183
Skills gi? nh? wanted-skill signal: 2


## 12. Áp dụng filtering cho have / want / target

In [13]:
def filter_skills(skills, valid_set):
    return [s for s in skills if s in valid_set]

model_df["have_skills"] = model_df["have_skills_raw"].apply(lambda x: filter_skills(x, skills_to_keep))
model_df["want_skills"] = model_df["want_skills_raw"].apply(lambda x: filter_skills(x, skills_to_keep))
model_df["target_skills"] = model_df.apply(
    lambda row: [s for s in row["want_skills"] if s not in set(row["have_skills"])],
    axis=1
)

model_df["n_have"] = model_df["have_skills"].apply(len)
model_df["n_want"] = model_df["want_skills"].apply(len)
model_df["n_target"] = model_df["target_skills"].apply(len)

print("Sau filtering:")
print("Users còn >=1 input skill:", int((model_df["n_have"] > 0).sum()))
print("Users có target skill:", int((model_df["n_target"] > 0).sum()))

display(model_df[["ResponseId", "have_skills", "want_skills", "target_skills"]].head(3))

Sau filtering:
Users còn >=1 input skill: 24229
Users có target skill: 16643


,ResponseId,have_skills,want_skills,target_skills
0,1,"[Language:Bash/Shell (all shells), Language:Da...",[Language:Dart],[]
1,2,"[Language:Java, Database:Dynamodb, Database:Mo...","[Language:Java, Language:Python, Language:Swif...","[Language:Python, Language:Swift]"
2,3,"[Language:Dart, Language:HTML/CSS, Language:Ja...","[Language:Dart, Language:HTML/CSS, Language:Ja...","[Database:Firebase Realtime Database, Platform..."


## 13. Tạo metadata vocabulary cuối cùng

In [14]:
final_have_counter = Counter()
final_want_counter = Counter()

for skills in model_df["have_skills"]:
    final_have_counter.update(skills)

for skills in model_df["want_skills"]:
    final_want_counter.update(skills)

skill_vocab_final = pd.DataFrame({
    "skill": sorted(skills_to_keep)
})
skill_vocab_final["have_count"] = skill_vocab_final["skill"].map(final_have_counter).fillna(0).astype(int)
skill_vocab_final["want_count"] = skill_vocab_final["skill"].map(final_want_counter).fillna(0).astype(int)
skill_vocab_final["have_freq"] = skill_vocab_final["have_count"] / len(model_df)
skill_vocab_final["want_freq"] = skill_vocab_final["want_count"] / len(model_df)
skill_vocab_final["category"] = skill_vocab_final["skill"].str.split(":").str[0]
skill_vocab_final["token"] = skill_vocab_final["skill"].str.split(":", n=1).str[1]

display(skill_vocab_final.sort_values("have_count", ascending=False).head(20))
print("Final vocabulary size:", len(skill_vocab_final))

,skill,have_count,want_count,have_freq,want_freq,category,token
91,Language:JavaScript,16417,8098,0.677576,0.334228,Language,JavaScript
67,DevEnv:Visual Studio Code,15411,9799,0.636056,0.404433,DevEnv,Visual Studio Code
89,Language:HTML/CSS,14994,8051,0.618845,0.332288,Language,HTML/CSS
107,Language:SQL,14697,8754,0.606587,0.361303,Language,SQL
124,Platform:Docker,14348,9928,0.592183,0.409757,Platform,Docker
103,Language:Python,12911,8629,0.532874,0.356143,Language,Python
39,Database:PostgreSQL,11887,9622,0.490610,0.397127,Database,PostgreSQL
110,Language:TypeScript,11687,8236,0.482356,0.339923,Language,TypeScript
74,Language:Bash/Shell (all shells),11615,6177,0.479384,0.254942,Language,Bash/Shell (all shells)
153,Platform:npm,11517,5334,0.475339,0.220149,Platform,npm


Final vocabulary size: 183


## 14. Tạo train / validation / test split theo USER

### Đây là bước chống data leakage
- rules chỉ học trên train
- graph chỉ build trên train
- test chỉ dùng để evaluate

In [15]:
# Chỉ giữ những user có ít nhất 1 input skill
split_df = model_df.loc[model_df["n_have"] > 0].copy()

train_df, temp_df = train_test_split(
    split_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Train: (19383, 189)
Validation: (2423, 189)
Test: (2423, 189)


## 15. Chuẩn bị dataset cho FP-Growth

In [16]:
transactions_train = train_df["have_skills"].tolist()
transactions_valid = valid_df["have_skills"].tolist()
transactions_test  = test_df["have_skills"].tolist()

print("Ví dụ transaction train:")
print(transactions_train[:2])

Ví dụ transaction train:
[['Language:C#', 'Language:HTML/CSS', 'Language:JavaScript', 'Language:SQL', 'Language:TypeScript', 'Language:Visual Basic (.Net)', 'Database:Microsoft SQL Server', 'Webframe:ASP.NET', 'Webframe:React', 'DevEnv:Notepad++', 'DevEnv:Visual Studio', 'DevEnv:Visual Studio Code'], ['Language:Assembly', 'Language:Bash/Shell (all shells)', 'Language:C', 'Language:Java', 'Language:JavaScript', 'Language:Perl', 'Language:PowerShell', 'Language:Python', 'Platform:Make', 'Platform:Maven (build tool)', 'Platform:Pacman', 'Platform:Pip', 'DevEnv:Android Studio', 'DevEnv:Eclipse', 'DevEnv:Notepad++', 'DevEnv:Visual Studio', 'DevEnv:Visual Studio Code']]


## 16. One-hot encoding cho FP-Growth

Lưu ý:
- chỉ fit encoder trên vocabulary train
- transform valid/test theo cùng encoder

In [17]:
te = TransactionEncoder()
train_ohe = te.fit(transactions_train).transform(transactions_train)
valid_ohe = te.transform(transactions_valid)
test_ohe  = te.transform(transactions_test)

X_train_fpg = pd.DataFrame(train_ohe, columns=te.columns_).astype(bool)
X_valid_fpg = pd.DataFrame(valid_ohe, columns=te.columns_).astype(bool)
X_test_fpg  = pd.DataFrame(test_ohe,  columns=te.columns_).astype(bool)

print("FP-Growth train matrix:", X_train_fpg.shape)
display(X_train_fpg.head())

FP-Growth train matrix: (19383, 183)


,AIModel:Alibaba Cloud Qwen models,AIModel:Amazon Titan models,AIModel:Anthropic: Claude Sonnet,AIModel:DeepSeek (R- Reasoning models),AIModel:DeepSeek (V- General purpose models),AIModel:Gemini (Flash general purpose models),AIModel:Gemini (Pro Reasoning models),AIModel:Meta Llama (all models),AIModel:Microsoft Phi-4 models,AIModel:Mistral AI models,...,Webframe:Nuxt.js,Webframe:Phoenix,Webframe:React,Webframe:Ruby on Rails,Webframe:Spring Boot,Webframe:Svelte,Webframe:Symfony,Webframe:Vue.js,Webframe:WordPress,Webframe:jQuery
0,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False


## 17. Chuẩn bị graph co-occurrence cho Node2Vec

Edge weight = số lần hai skill cùng xuất hiện trong train users.

In [18]:
def build_cooccurrence_graph(transactions, min_edge_weight=2):
    edge_counter = Counter()
    node_counter = Counter()

    for skills in transactions:
        unique_skills = sorted(set(skills))
        node_counter.update(unique_skills)
        for a, b in itertools.combinations(unique_skills, 2):
            edge_counter[(a, b)] += 1

    G = nx.Graph()
    for node, cnt in node_counter.items():
        G.add_node(node, support=cnt)

    for (a, b), w in edge_counter.items():
        if w >= min_edge_weight:
            G.add_edge(a, b, weight=w)

    return G, node_counter, edge_counter

G_train, train_node_counter, train_edge_counter = build_cooccurrence_graph(transactions_train, min_edge_weight=2)

print("Graph nodes:", G_train.number_of_nodes())
print("Graph edges:", G_train.number_of_edges())

Graph nodes: 183
Graph edges: 16653


## 18. Chuẩn bị dataset evaluation

Mỗi user cần:
- input_skills
- target_skills

Target = `want_skills - have_skills`

In [19]:
eval_train = train_df[["ResponseId", "have_skills", "target_skills", "is_junior_0_3y"]].copy()
eval_valid = valid_df[["ResponseId", "have_skills", "target_skills", "is_junior_0_3y"]].copy()
eval_test  = test_df[["ResponseId", "have_skills", "target_skills", "is_junior_0_3y"]].copy()

print("Users có target ở train:", int((eval_train["target_skills"].apply(len) > 0).sum()))
print("Users có target ở valid:", int((eval_valid["target_skills"].apply(len) > 0).sum()))
print("Users có target ở test:", int((eval_test["target_skills"].apply(len) > 0).sum()))

display(eval_test.head())

Users có target ở train: 13311
Users có target ở valid: 1630
Users có target ở test: 1702


,ResponseId,have_skills,target_skills,is_junior_0_3y
0,20835,"[Language:Assembly, Language:Kotlin, Language:...",[],False
1,10164,"[Language:Bash/Shell (all shells), Language:C,...",[AIModel:openAI Image generating models],False
2,44230,"[Language:Bash/Shell (all shells), Language:C,...","[Language:Go, Language:Rust]",False
3,26035,"[Language:C++, Language:JavaScript, Language:K...",[],False
4,34410,"[Language:Bash/Shell (all shells), Language:HT...","[Platform:Terraform, DevEnv:IntelliJ IDEA]",False


## 19. Xuất file sạch để dùng cho các notebook training sau

### File xuất ra:
1. `cleaned_model_population.csv`
2. `skill_vocabulary_final.csv`
3. `train_users.csv`
4. `valid_users.csv`
5. `test_users.csv`
6. `fpgrowth_train_matrix.csv`
7. `graph_edges_train.csv`

In [20]:
def list_to_pipe_string(items):
    return "|".join(items)

export_cols = [
    "ResponseId",
    "MainBranch",
    "Employment",
    "WorkExp",
    "WorkExp_num",
    "DevType",
    "is_junior_0_3y",
    "have_skills",
    "want_skills",
    "target_skills",
    "n_have",
    "n_want",
    "n_target",
]

export_df = model_df[export_cols].copy()
for c in ["have_skills", "want_skills", "target_skills"]:
    export_df[c] = export_df[c].apply(list_to_pipe_string)

train_export = train_df[export_cols].copy()
valid_export = valid_df[export_cols].copy()
test_export  = test_df[export_cols].copy()

for temp in [train_export, valid_export, test_export]:
    for c in ["have_skills", "want_skills", "target_skills"]:
        temp[c] = temp[c].apply(list_to_pipe_string)

# Export ??ng graph ?? l?c b?ng min_edge_weight trong build_cooccurrence_graph().
graph_edges_df = pd.DataFrame(
    [(a, b, data.get("weight", 1)) for a, b, data in G_train.edges(data=True)],
    columns=["skill_a", "skill_b", "weight"]
).sort_values("weight", ascending=False)

export_df.to_csv("cleaned_model_population.csv", index=False)
skill_vocab_final.to_csv("skill_vocabulary_final.csv", index=False)
train_export.to_csv("train_users.csv", index=False)
valid_export.to_csv("valid_users.csv", index=False)
test_export.to_csv("test_users.csv", index=False)
X_train_fpg.to_csv("fpgrowth_train_matrix.csv", index=False)
graph_edges_df.to_csv("graph_edges_train.csv", index=False)

print("Đã export xong các file sạch.")

Đã export xong các file sạch.


## 20. Download toàn bộ file output

In [ ]:
import os

output_files = [
    "cleaned_model_population.csv",
    "skill_vocabulary_final.csv",
    "train_users.csv",
    "valid_users.csv",
    "test_users.csv",
    "fpgrowth_train_matrix.csv",
    "graph_edges_train.csv",
]

for f in output_files:
    if os.path.exists(f):
        print(f"✔ File sẵn sàng: {os.path.abspath(f)}")
    else:
        print(f"✘ Không tìm thấy: {f}")

✔ File sẵn sàng: d:\School\BTVN\BDM\fn\cleaned_model_population.csv
✔ File sẵn sàng: d:\School\BTVN\BDM\fn\skill_vocabulary_final.csv
✔ File sẵn sàng: d:\School\BTVN\BDM\fn\train_users.csv
✔ File sẵn sàng: d:\School\BTVN\BDM\fn\valid_users.csv
✔ File sẵn sàng: d:\School\BTVN\BDM\fn\test_users.csv
✔ File sẵn sàng: d:\School\BTVN\BDM\fn\fpgrowth_train_matrix.csv
✔ File sẵn sàng: d:\School\BTVN\BDM\fn\graph_edges_train.csv


## 21. Tiêu chí hoàn tất bước preprocessing

Sau notebook này, bạn đã có đầy đủ dữ liệu cho:

### FP-Growth
- `fpgrowth_train_matrix.csv`
- hoặc `train_users.csv` với cột `have_skills`

### Node2Vec
- `graph_edges_train.csv`

### Evaluation
- `valid_users.csv`
- `test_users.csv`

---

## Bước tiếp theo
Notebook tiếp theo nên làm 3 phần:
1. Train FP-Growth + sinh association rules  
2. Train Node2Vec trên graph kỹ năng  
3. Hybrid ranking + evaluation Recall@K / HitRate@K / Coverage